# Fase 4 — Formateo final para el Splitter
### Puente entre Enriquecimiento (metadatos financieros) y Chunking (`SentenceSplitter`)

**Posición en el pipeline:** este notebook se ejecuta sobre el JSON central **ya
enriquecido** (con `ticker` y demás metadatos de YahooFinance ya inyectados como
claves del diccionario). Su única responsabilidad es dejar el string `contenido` de
cada noticia en su forma tipográfica final:

1. Normalización tipográfica (comillas rectas, guion estándar).
2. Eliminación de la sintaxis Markdown de los encabezados (conservando su texto).
3. Fusión de saltos de línea simples que cortan una frase por la mitad.
4. **Regla de oro:** los `\n\n` (separadores de párrafo) no se tocan — el
   `SentenceSplitter` de la fase siguiente depende de ellos.
5. Inyección de una cabecera de contexto global (`Título` + `Entidad relacionada`)
   al principio del texto.

No se repite aquí la limpieza estructural de ruido de scraping (menús de navegación,
widgets, boilerplate de portal) porque esa responsabilidad ya la cubrió la Fase 1
(`limpiador.py` / `fase1_limpieza_estructural.ipynb`) antes del enriquecimiento.


## 1. 📤 Carga manual del archivo JSON

Sube directamente el archivo JSON central ya enriquecido con metadatos financieros
(salida de la Fase 3) — sin necesidad de tenerlo ya en Google Drive. Esto define
`INPUT_JSON` y deriva `OUTPUT_JSON` a partir del nombre subido.


In [ ]:
# --- Detección de entorno (Colab vs. local) ----------------------------------
try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    print("No estamos en Google Colab: se omite el selector de subida.")

import json
import re
import os

if IN_COLAB:
    print("Sube tu archivo JSON central (salida de la Fase 3 de enriquecimiento):")
    subido = files.upload()
    if subido:
        INPUT_JSON = list(subido.keys())[0]
        print(f"Archivo recibido: {INPUT_JSON}")
    else:
        raise RuntimeError("No se subió ningún archivo. Vuelve a ejecutar la celda para intentarlo de nuevo.")
else:
    # Fuera de Colab: usa un archivo local ya presente en el directorio actual.
    INPUT_JSON = "noticias_central.json"
    if not os.path.exists(INPUT_JSON):
        raise FileNotFoundError(
            f"No se encontró '{INPUT_JSON}' localmente. Coloca tu archivo de entrada "
            f"en el directorio actual o ajusta la variable INPUT_JSON manualmente."
        )
    print(f"Usando archivo local: {INPUT_JSON}")

# OUTPUT_JSON se deriva del nombre del archivo de entrada, para que quede
# claro a qué lote corresponde el resultado formateado.
_nombre_base, _ext = os.path.splitext(INPUT_JSON)
if not _ext:
    _ext = ".json"
OUTPUT_JSON = f"{_nombre_base}_fase4{_ext}"

print(f"INPUT_JSON  = {INPUT_JSON}")
print(f"OUTPUT_JSON = {OUTPUT_JSON}")


## 2. Función core de formateo e inyección (CRÍTICA)

Se construye en cuatro piezas independientes (una por requisito) y luego se orquestan
en `formatear_contenido()`. Cada pieza se puede probar/ajustar por separado.


### 2.1 Normalización tipográfica

Comillas curvas dobles (`“ ”`) y simples (`‘ ’`) → comillas rectas. Guion largo (`—`,
*em dash*) y guion medio (`–`, *en dash*) → guion estándar (`-`). Esta sustitución es
un simple cambio de carácter y **no toca ningún `\n`**, por lo que es segura de
aplicar en cualquier punto del pipeline sin afectar a la estructura de párrafos.


In [ ]:
QUOTE_MAP = {
    "\u201c": '"', "\u201d": '"',   # comillas dobles tipográficas “ ”
    "\u2018": "'", "\u2019": "'",   # comillas simples tipográficas ‘ ’
}
DASH_MAP = {
    "\u2014": "-",   # guion largo (em dash) —
    "\u2013": "-",   # guion medio (en dash) –
}

def normalizar_tipografia(text: str) -> str:
    """Sustituye comillas curvas por comillas rectas y guiones largos/medios
    por el guion estándar '-'. No modifica ningún salto de línea."""
    if not text:
        return text
    for curva, recta in QUOTE_MAP.items():
        text = text.replace(curva, recta)
    for guion_tipografico, guion_estandar in DASH_MAP.items():
        text = text.replace(guion_tipografico, guion_estandar)
    return text


### 2.2 Eliminación de sintaxis Markdown

Se retira únicamente el símbolo (`#`, `##`, `###`...) al inicio de línea. El texto del
encabezado **se conserva**, quedando como un párrafo normal — no se borra la
información, solo la marca de formato que ya no aporta nada de cara al embedding.


In [ ]:
MARKDOWN_HEADER_RE = re.compile(r"^[ \t]*#{1,6}[ \t]+", re.MULTILINE)

def eliminar_markdown_headers(text: str) -> str:
    """Elimina la sintaxis '#'/'##'/'###'... al inicio de línea, dejando el
    texto del encabezado como un párrafo normal (no se pierde información)."""
    if not text:
        return text
    return MARKDOWN_HEADER_RE.sub("", text)


### 2.3 Fusión de líneas huérfanas + REGLA DE ORO

Un `\n` **simple** (aislado, sin otro `\n` inmediatamente antes ni después) es casi
siempre ruido de *scraping*: el HTML se rompió en mitad de una frase. Ese se sustituye
por un espacio.

Un `\n\n` (o más) es la señal de párrafo real que usará el `SentenceSplitter` — **no
se toca en ningún caso**.

La distinción se resuelve con una única expresión regular basada en *lookaround*:

```python
re.sub(r"(?<!\n)\n(?!\n)", " ", text)
```

- `(?<!\n)`: el `\n` candidato **no** puede estar precedido por otro `\n`.
- `(?!\n)`: el `\n` candidato **no** puede estar seguido por otro `\n`.

Con estas dos condiciones, **ningún** carácter `\n` que forme parte de un `\n\n` (o de
una racha de 3+) puede hacer *match* nunca — solo se sustituyen los `\n` verdaderamente
aislados. Esto garantiza la regla de oro por construcción, no por casualidad.


In [ ]:
def fusionar_lineas_huerfanas(text: str) -> str:
    """Sustituye por un espacio los saltos de línea SIMPLES y aislados
    (ruido de scraping en mitad de una frase), sin tocar nunca los '\n\n'
    que separan párrafos reales.

    REGLA DE ORO: ningún carácter que forme parte de un '\n\n' (o de una
    racha de 3+ '\n') puede hacer match con este patrón, por construcción
    del lookaround -- no depende de un post-procesado que podría fallar.
    """
    if not text:
        return text
    text = re.sub(r"(?<!\n)\n(?!\n)", " ", text)
    # higiene: la fusión anterior puede dejar espacios dobles sueltos
    # (p. ej. si la línea original terminaba en espacio antes del '\n').
    text = re.sub(r"[ \t]{2,}", " ", text)
    return text


### 2.4 Inyección de contexto global

Cabecera con el formato exacto solicitado, antepuesta al texto ya formateado:

```
CONTEXTO GLOBAL - Título: [titulo] | Entidad relacionada: [ticker]

```

El `ticker` se lee directamente del diccionario de metadatos de la noticia (la clave
que añadió el enriquecimiento con YahooFinance). Si faltara alguno de los dos campos,
se usa `"N/D"` en vez de fallar.


In [ ]:
def inyectar_contexto(texto_formateado: str, metadatos: dict) -> str:
    """Antepone la cabecera de contexto global exacta, seguida de '\n\n'.

    El título se pasa también por normalizar_tipografia() (viene del campo
    'titulo' en crudo, que puede traer sus propias comillas curvas o
    guiones tipográficos) y se colapsa cualquier salto de línea que
    pudiera traer -- el título es, por definición, una única línea dentro
    de la cabecera de contexto, así que un '\n' suelto ahí no es un
    separador de párrafo legítimo."""
    titulo = normalizar_tipografia((metadatos.get("titulo") or "N/D").strip())
    titulo = re.sub(r"\s*\n+\s*", " ", titulo)  # el título es siempre una sola línea
    ticker = (metadatos.get("ticker") or "N/D")
    ticker = ticker.strip() if isinstance(ticker, str) else str(ticker)

    cabecera = f"CONTEXTO GLOBAL - Título: {titulo} | Entidad relacionada: {ticker}\n\n"
    return cabecera + texto_formateado


### 2.5 Función maestra: `formatear_contenido`

Orquesta las cuatro piezas anteriores en el orden correcto.

In [ ]:
def formatear_contenido(texto: str, metadatos: dict) -> str:
    """Función core de la Fase 4.

    Parámetros
    ----------
    texto : str
        El campo 'contenido' de la noticia (ya limpio de ruido estructural
        por la Fase 1, ya con metadatos financieros disponibles en el dict).
    metadatos : dict
        El diccionario completo de la noticia (se usa para leer 'titulo' y
        'ticker' al inyectar la cabecera de contexto).

    Orden de aplicación (importa):
      1. Normalización tipográfica (no toca '\n', segura en cualquier punto).
      2. Eliminación de sintaxis Markdown (line-based, tampoco altera '\n\n').
      3. Fusión de líneas huérfanas -- aplicada DESPUÉS de los pasos anteriores
         para que ya no queden '#' ni comillas curvas que puedan interferir
         con los patrones de limpieza.
      4. Inyección de la cabecera de contexto (al final, como último párrafo
         añadido por delante).
    """
    texto = texto or ""

    texto = normalizar_tipografia(texto)
    texto = eliminar_markdown_headers(texto)
    texto = fusionar_lineas_huerfanas(texto)
    texto = texto.strip()

    return inyectar_contexto(texto, metadatos)


## 3. Verificación rápida sobre un ejemplo

In [ ]:
ejemplo_metadatos = {
    "titulo": "Wall Street revive con la rebaja de \u2018S&P\u2019",
    "ticker": "^GSPC",
}

ejemplo_contenido = (
    "\u201cEsto es solo el principio\u201d, dijo un analista de Nueva\n"
    "York tras conocerse la rebaja \u2014en plena madrugada\u2014 de la calificaci\u00f3n "
    "crediticia.\n\n"
    "## Impacto en Europa\n\n"
    "Los mercados europeos abrieron\ncon fuertes ca\u00eddas, arrastrados por el "
    "pesimismo\ngeneral."
)

resultado = formatear_contenido(ejemplo_contenido, ejemplo_metadatos)
print(repr(resultado))
print()
print("--- versión legible ---")
print(resultado)
print()

# Comprobaciones automáticas
assert resultado.startswith("CONTEXTO GLOBAL - Título:"), "falta la cabecera de contexto"
assert "Entidad relacionada: ^GSPC" in resultado, "falta el ticker en la cabecera"
assert "\u201c" not in resultado and "\u201d" not in resultado, "quedan comillas curvas dobles"
assert "\u2018" not in resultado and "\u2019" not in resultado, "quedan comillas curvas simples"
assert "\u2014" not in resultado and "\u2013" not in resultado, "queda un guion largo/medio sin normalizar"
assert not re.search(r"^[ \t]*#{1,6}[ \t]", resultado, re.MULTILINE), "quedan encabezados Markdown"
assert "\n\n" in resultado, "se perdió algún separador de párrafo real"
assert "Nueva York" in resultado, "no se fusionó correctamente la línea huérfana 'Nueva\\nYork'"
assert "abrieron con fuertes caídas" in resultado, "no se fusionó correctamente la segunda línea huérfana"

print("Todas las verificaciones pasaron correctamente.")


## 4. Procesamiento del JSON central

Carga el único archivo JSON central (lista de diccionarios), aplica
`formatear_contenido` al campo `contenido` de cada noticia (usando el propio
diccionario de la noticia como `metadatos`), actualiza esa clave y guarda el
resultado completo en `OUTPUT_JSON`. El resto de campos (`url`, `fecha`, `ticker`,
métricas de YahooFinance, etc.) se conservan sin modificar.

Incluye manejo de errores por registro: si una noticia individual falla al
formatearse, se conserva con su `contenido` original (sin perderla) y se registra la
incidencia, en vez de detener todo el proceso.


In [ ]:
# --- Carga del JSON central ---------------------------------------------------
with open(INPUT_JSON, encoding="utf-8") as f:
    noticias = json.load(f)

if not isinstance(noticias, list):
    raise ValueError(
        "Se esperaba una LISTA de diccionarios en el JSON central; "
        f"se encontró un {type(noticias).__name__}."
    )

print(f"Noticias cargadas: {len(noticias)}")

# --- Procesamiento --------------------------------------------------------------
errores = []
noticias_formateadas = []

for idx, noticia in enumerate(noticias):
    if not isinstance(noticia, dict):
        errores.append({"indice": idx, "tipo": "registro_no_es_objeto"})
        noticias_formateadas.append(noticia)
        continue
    try:
        contenido_original = noticia.get("contenido", "") or ""
        contenido_formateado = formatear_contenido(contenido_original, noticia)

        nueva_noticia = dict(noticia)
        nueva_noticia["contenido"] = contenido_formateado
        noticias_formateadas.append(nueva_noticia)
    except Exception as e:
        errores.append({
            "indice": idx,
            "titulo": (noticia.get("titulo") or "")[:80],
            "tipo": "fallo_formateo",
            "detalle": str(e),
        })
        noticias_formateadas.append(noticia)  # se conserva sin formatear, mejor que perderla

print(f"Noticias formateadas correctamente: {len(noticias_formateadas) - len(errores)}")
print(f"Incidencias registradas:            {len(errores)}")

if errores:
    print("\nDetalle de incidencias:")
    for e in errores[:20]:
        print(" -", e)


### 4.1 Guardado del JSON de salida

Se conserva la codificación UTF-8 real (`ensure_ascii=False`).

In [ ]:
with open(OUTPUT_JSON, "w", encoding="utf-8") as f:
    json.dump(noticias_formateadas, f, ensure_ascii=False, indent=2)

print(f"Guardado: {OUTPUT_JSON}  ({len(noticias_formateadas)} noticias)")

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_JSON)
else:
    print("Fuera de Colab: descarga el archivo directamente desde el sistema de ficheros.")


### 4.2 Verificación sobre el corpus completo

Comprueba, sobre TODOS los registros formateados, que las cuatro reglas se cumplieron
de forma consistente (no solo en el ejemplo de la Sección 3).


In [ ]:
problemas = {
    "comillas_curvas": 0,
    "guion_tipografico": 0,
    "markdown_header_residual": 0,
    "salto_simple_residual": 0,
    "sin_cabecera_contexto": 0,
}

for n in noticias_formateadas:
    c = n.get("contenido", "") or ""
    if "\u201c" in c or "\u201d" in c or "\u2018" in c or "\u2019" in c:
        problemas["comillas_curvas"] += 1
    if "\u2014" in c or "\u2013" in c:
        problemas["guion_tipografico"] += 1
    if re.search(r"^[ \t]*#{1,6}[ \t]", c, re.MULTILINE):
        problemas["markdown_header_residual"] += 1
    if re.search(r"(?<!\n)\n(?!\n)", c):
        problemas["salto_simple_residual"] += 1
    if not c.startswith("CONTEXTO GLOBAL - Título:"):
        problemas["sin_cabecera_contexto"] += 1

print("Verificación sobre el corpus completo (deberían ser todo ceros):")
for nombre, n in problemas.items():
    estado = "OK" if n == 0 else "REVISAR"
    print(f"  [{estado}] {nombre}: {n}")


## 5. Resumen de las reglas aplicadas

| # | Requisito | Función | Técnica |
|---|---|---|---|
| 1 | Comillas tipográficas → rectas; guiones → `-` | `normalizar_tipografia` | Sustitución directa de caracteres, no toca `\n` |
| 2 | Eliminar sintaxis Markdown | `eliminar_markdown_headers` | `re.sub(r"^[ \t]*#{1,6}[ \t]+", "", text, flags=re.MULTILINE)` — se conserva el texto |
| 3 | Fusionar líneas huérfanas | `fusionar_lineas_huerfanas` | `re.sub(r"(?<!\n)\n(?!\n)", " ", text)` |
| 4 (regla de oro) | Preservar `\n\n` | — | Garantizado por construcción: el *lookaround* del punto 3 nunca hace match dentro de una racha de 2+ `\n` |
| 5 | Inyección de contexto | `inyectar_contexto` | `"CONTEXTO GLOBAL - Título: ... \| Entidad relacionada: ...\n\n"` + texto |

**Fuera de alcance de esta fase** (ya resuelto antes en el pipeline): limpieza de
ruido estructural de scraping (menús, widgets, boilerplate de portal, mojibake) — eso
es responsabilidad de la Fase 1. La Fase 4 asume que `contenido` ya llega limpio de
ese tipo de ruido y solo necesita el formateo tipográfico final antes del chunking.


## 6. 📥 Descarga de resultados

Descarga a tu ordenador el JSON con el `contenido` ya formateado e inyectado con la
cabecera de contexto global, listo para la fase de *Chunking* con `SentenceSplitter`.


In [ ]:
if IN_COLAB:
    print(f"Descargando {OUTPUT_JSON} ...")
    files.download(OUTPUT_JSON)
else:
    print("Fuera de Colab: el archivo ya está guardado en el directorio actual; "
          f"descárgalo directamente desde ahí ({OUTPUT_JSON}).")
